# Track B — Fase 0: Scaffold & Sanity
**BDC Satria Data 2026 — Klasifikasi Citra Sampah**

Notebook orchestrator TIPIS — semua logika ada di `track_b/src/*.py` (version-controlled di Git).
Tidak ada `%%writefile` di sini; edit kode = edit file di `src/`, bukan sel notebook.

Isi:
1. Environment check (GPU, library)
2. Setup path ke `src/` (clone repo di Colab / relative path lokal)
3. Cek per modul: seed, model, loader stub, loss/metric
4. Smoke test training loop 1 epoch (data dummy)
5. **Sanity overfit 1 batch** — milestone B1: loss HARUS ≈ 0

> ⚠️ Versi sanity sebelumnya loss stuck di ~1.06 (≈ ln 3 = tebakan acak) tapi tetap print "lolos"
> karena assertion-nya lemah. `sanity_overfit.py` sudah diperbaiki: LR 3e-3→3e-4, clip 0.5→1.0,
> drop_path 0, dan assert `final_loss < 0.1`. **Wajib rerun sampai hijau sebelum Fase 1.**

In [ ]:
# === 1. Environment Check ===
!pip install timm scikit-learn -q

import torch
import timm
import sklearn
import os

print("=" * 50)
print("ENVIRONMENT CHECK")
print("=" * 50)
print(f"torch:    {torch.__version__}")
print(f"cuda:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu:      {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"vram:     {vram:.1f} GB")
print(f"timm:     {timm.__version__}")
print(f"sklearn:  {sklearn.__version__}")
print(f"cpu cores: {os.cpu_count()}")
print("=" * 50)

assert torch.cuda.is_available(), "GPU tidak aktif"
print("GPU aktif")

^C


ModuleNotFoundError: No module named 'torch'

In [ ]:
# === 2. Setup path ke src/ ===
import sys
from pathlib import Path

if Path("../src/model.py").exists():
    # Lokal: notebook dijalankan dari track_b/notebooks/
    SRC = Path("../src").resolve()
else:
    # Colab: clone repo (sekali per session)
    if not Path("satria-data-bdcugm02").exists():
        !git clone https://github.com/agaggigit/satria-data-bdcugm02.git
    SRC = Path("satria-data-bdcugm02/track_b/src").resolve()

sys.path.insert(0, str(SRC))
print(f"src path: {SRC}")
assert (SRC / "sanity_overfit.py").exists(), "src/ tidak ditemukan — cek clone/path"
print("Modul src/ siap di-import")

In [ ]:
# === 3a. Cek seed reproducibility ===
import torch
from seed_utils import set_seed

set_seed(42)
t1 = torch.randn(3)
set_seed(42)
t2 = torch.randn(3)
assert torch.equal(t1, t2), "Seed tidak reproducible"
print("seed_utils OK — reproducibility terverifikasi")

In [ ]:
# === 3b. Cek model + data config ===
import torch
from model import build_model, get_data_config

model = build_model().cuda()
data_config = get_data_config(model)

dummy = torch.randn(4, 3, 224, 224).cuda()
with torch.autocast(device_type="cuda", dtype=torch.float16):
    out = model(dummy)

print(f"output shape: {out.shape}")
print(f"params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"mean: {data_config['mean']}")
print(f"std:  {data_config['std']}")
assert out.shape == (4, 3), f"Shape salah: {out.shape}"

del model, dummy, out
torch.cuda.empty_cache()
print("Model OK — forward pass benar")

In [ ]:
# === 3c. Cek loader stub (kontrak Track A) ===
from dataset_stub import get_loaders

train_loader, val_loader = get_loaders(fold=0, img_size=224, batch=32)
images, labels = next(iter(train_loader))

print(f"images shape: {images.shape}")
print(f"labels:       {labels[:8]}")
assert images.shape[1:] == (3, 224, 224), f"Shape salah: {images.shape}"
assert labels.min() >= 0 and labels.max() <= 2, "Label di luar 0/1/2"
print(f"train batches: {len(train_loader)} | val batches: {len(val_loader)}")
print("Loader stub OK — signature sesuai kontrak")

In [ ]:
# === 3d. Cek loss + metric ===
import torch
from losses_metrics import build_loss, macro_f1, print_report

device = "cuda"
criterion = build_loss(torch.tensor([1.0, 1.0, 1.0]).to(device))

logits = torch.randn(9, 3).to(device)
labels = torch.tensor([0, 1, 2, 0, 1, 2, 0, 1, 2]).to(device)

loss = criterion(logits, labels)
print(f"loss: {loss.item():.4f}")
assert loss.item() > 0, "Loss harus positif"

preds = logits.argmax(dim=1)
f1 = macro_f1(preds, labels)
print(f"macro-f1: {f1:.4f}")
assert 0 <= f1 <= 1, "F1 di luar range"

print_report(preds, labels)
print("Loss + metric OK")

In [ ]:
# === 4. Smoke test training loop — 1 epoch data dummy ===
from train import run_training

f1 = run_training(fold=0, epochs=1, batch=32)
print(f"\nTraining loop jalan end-to-end, F1 (dummy): {f1:.4f}")

In [ ]:
# === 5. SANITY OVERFIT 1 BATCH — milestone B1 ===
# Kriteria: model menghafal 8 gambar → final loss < 0.1.
# Kalau assert gagal, loop BELUM terbukti benar — jangan lanjut Fase 1.
import sys
for mod in ["sanity_overfit", "model", "losses_metrics", "seed_utils"]:
    sys.modules.pop(mod, None)

from sanity_overfit import sanity_overfit
final_loss = sanity_overfit()

print(f"\nFase 0 SELESAI — milestone B1 tercapai (final loss {final_loss:.4f})")
print("Siap terima folds.csv + dataset.py + class_weights dari Track A")

## Checklist Fase 0

- [ ] Env siap (timm/torch, GPU terdeteksi)
- [ ] Seed reproducible
- [ ] Model forward pass benar (`[B, 3]`)
- [ ] Loader stub sesuai kontrak `get_loaders(fold, img_size, batch)`
- [ ] Loss (weighted CE + smoothing) & Macro-F1 benar
- [ ] Training loop 1 epoch jalan di data dummy
- [ ] **Sanity overfit lolos dengan final loss < 0.1** ← kriteria B1 yang sebenarnya

**Next:** `02_baseline_fold0.ipynb` (Fase 1) — setelah GATE 2 Track A hijau.